# Stage-A Dataset Quickstart (Colab)

This notebook unpacks the uploaded `Pipeline_stage_a.zip`, installs the runtime dependencies, runs the local test suite, generates a small dataset, and exercises `FullPipeline.forward` on GPU.

Expected upload: a zip produced from the repository root that contains, at minimum, the folders `PointPillars_module/`, `create_dataset_module/`, `urdf/`, and the file `pybullet_navigation.py`. See `create_dataset_module/README.md` for the exact `zip` / `Compress-Archive` command.

Runtime: **GPU (T4 or better)** recommended so that `PointPillarsNeckExtractor` and `mamba-ssm` can be exercised end-to-end.

## 1. Upload the zip and unpack it

In [ ]:
import os, sys, shutil, pathlib, zipfile

from google.colab import files
uploaded = files.upload()
assert uploaded, 'Please upload Pipeline_stage_a.zip'
zip_name = next(iter(uploaded))
work_root = pathlib.Path('/content/Pipeline')
if work_root.exists():
    shutil.rmtree(work_root)
work_root.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(work_root)
os.chdir(work_root)
print('Working dir:', os.getcwd())
print('Contents  :', sorted(os.listdir('.')))


## 2. Install runtime dependencies

`mamba-ssm` only compiles on Linux + CUDA. Colab fits that bill; if it still fails, `MambaTemporal` automatically falls back to `nn.GRU` and the rest of the pipeline keeps working.

In [ ]:
!pip -q install pybullet matplotlib
try:
    import mamba_ssm  # noqa: F401
    print('mamba-ssm already installed')
except ImportError:
    !pip -q install causal-conv1d
    !pip -q install mamba-ssm


## 3. Build the PointPillars CUDA extension (voxel_op)

`PointPillarsNeckExtractor` needs the `voxel_op` extension shipped with the `pointpillars` Python package. On Colab we install it from the vendored source tree inside `PointPillars_module/`.

In [ ]:
import subprocess
candidates = [
    'PointPillars_module/pointpillars',  # editable source tree, if included
    'PointPillars_module',               # fallback: package lives at top level
]
setup_dir = None
for c in candidates:
    if os.path.isfile(os.path.join(c, 'setup.py')) or os.path.isfile(os.path.join(c, 'pyproject.toml')):
        setup_dir = c
        break
if setup_dir is None:
    print('pointpillars source not found in the zip; skipping compile step.')
    print('Stage A dataset loading does NOT need voxel_op. You can still run Stage 4/5 below.')
else:
    print('Installing pointpillars from', setup_dir)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', setup_dir])


## 4. Run the test suite

All 15 `create_dataset_module` tests plus the `PointPillars_module` tests should pass on Colab.

In [ ]:
!python -m unittest discover -s create_dataset_module/tests -t . -v


## 5. Generate a small dataset + run FullPipeline.forward

`verify.py` does the complete loop: PyBullet rollout, `Trajectory.npz`, `RiskDataset`, collate, and (if a PointPillars checkpoint is available) `FullPipeline.forward` on CUDA.

In [ ]:
!python -m create_dataset_module.verify --tmp_dir ./_verify_tmp --keep


## 6. Scale up

Edit the cell below to produce the full training dataset. Rule of thumb: 200 scenes x 4 rollouts x 400 frames ~= 320k indexable samples, which is ample for Stage A.

In [ ]:
from create_dataset_module import DataGenerator
from create_dataset_module.config import DataGenConfig

cfg = DataGenConfig(
    out_dir='./data/stage_a',
    n_scenes=8,
    rollouts_per_scene=2,
    frames_per_rollout=200,
    policy_random_p=0.5,
    policy_scripted_p=0.3,
    policy_adversarial_p=0.2,
    seed=0,
)
n = DataGenerator(cfg).run()
print('wrote', n, 'rollouts to', cfg.out_dir)


## 7. (Optional) Pack the dataset for Drive

Save the generated `.npz` tree to Google Drive so you can resume training across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/RobotDog/stage_a
!cp -r ./data/stage_a/* /content/drive/MyDrive/RobotDog/stage_a/
print('pushed to /content/drive/MyDrive/RobotDog/stage_a')
